# Week 8 — Evaluation Expansion: Beyond R²

**Handbook deliverable:** metrics beyond R² (MAPE, MdAPE) and a price-band breakdown,
plus `metrics_summary.csv`.

**Scope note ** This notebook does ONLY the handbook Week 8 content.
It does not retrain any model, does not tune anything, and does not select a new
champion. The champion was already decided in Week 7: the tuned LightGBM
(depth 8, lr 0.1, n_estimators 800, test R² 0.8911), which beat the Week 6 RF-new
(test R² 0.8617). Everything here reuses the **saved test predictions** from
Weeks 6–7, so no new test-set scoring for selection happens in this notebook —
this is descriptive evaluation of a decision already made, not a second bite at
the test set.

**Numbering note.** The handbook calls this file `06_evaluation.ipynb`, but the
`06_` slot is already taken by the Week 7 notebook (`06_advanced_models.ipynb`).
This file is therefore named `07_evaluation.ipynb`.

**What this notebook does**
1. Loads the Week 7 champion's saved test predictions and the Week 6 RF-new
   saved test predictions, against the unchanged Week 3 test target.
2. Reports metrics beyond R² for both models: RMSE, MAE, MAPE, **MdAPE**
   (median absolute percentage error — the robust counterpart to MAPE, which a
   few huge errors can drag), median residual, and the ±10% hit rate.
3. Breaks the champion's errors down by **price band**, so a single headline
   number cannot hide a segment (e.g. luxury) that is systematically off.
4. Saves `metrics_summary.csv` and a one-page summary to `output_week8/`.

**Frozen Week 7 numbers to check against:** tuned LightGBM test R² 0.8911 /
RMSE 327,782 / MAE 167,600 / MAPE 13.0747%; RF-new test R² 0.8617 /
RMSE 369,380 / MAE 179,843 / MAPE 13.7617%; test n = 12,339.


## 0. Setup — load saved test predictions (no refitting)

Inputs (same folder layout as Weeks 3–7, folders next to this notebook):
- `output_week3/y_test.npy` — the test target, unchanged since Week 3
- `output_week7/gbm_tuned_test_pred.npy` — Week 7 champion test predictions
- `output_week6/rf_new_test_pred.npy` — Week 6 RF-new test predictions (comparison)
- `output_week7/model_comparison_week7.csv` (fallback) — frozen Week 7 test table,
  used only if a predictions file is missing


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

# ---- CONFIG: folders live next to this notebook (same layout as Weeks 3-7) ----
WEEK3_DIR = Path('output_week3')
WEEK6_DIR = Path('output_week6')
WEEK7_DIR = Path('output_week7')
OUT_DIR = Path('output_week8')
OUT_DIR.mkdir(exist_ok=True)

y_test = np.load(WEEK3_DIR / 'y_test.npy')
preds = {}
for name, path in [('LightGBM (tuned) — Week 7 champion', WEEK7_DIR / 'gbm_tuned_test_pred.npy'),
                   ('RF (new feats) — Week 6 champion', WEEK6_DIR / 'rf_new_test_pred.npy')]:
    if path.exists():
        preds[name] = np.load(path)
        print(f'{name}: loaded {path} ({len(preds[name])} rows)')
    else:
        print(f'{name}: MISSING {path} — that model will be reported from the frozen Week 7 table only')

print(f'Test target: {len(y_test)} rows')
assert all(len(p) == len(y_test) for p in preds.values())
assert not np.isnan(y_test).any()
for name, p in preds.items():
    assert not np.isnan(p).any(), f'NaNs in {name} predictions'
print('All loaded predictions match the test target length, no NaNs.')


LightGBM (tuned) — Week 7 champion: loaded output_week7/gbm_tuned_test_pred.npy (12339 rows)
RF (new feats) — Week 6 champion: loaded output_week6/rf_new_test_pred.npy (12339 rows)
Test target: 12339 rows
All loaded predictions match the test target length, no NaNs.


## 1. Metrics beyond R²

- **MAPE** is the mean absolute percentage error — what Weeks 5–7 already reported.
- **MdAPE** is the *median* absolute percentage error. Because it takes the middle
  house instead of the average, one $98M-style blow-up (Week 6) cannot drag it
  the way it drags MAPE. If MdAPE is much smaller than MAPE, the typical house is
  predicted better than the mean suggests, and a long error tail is doing damage.
- **Median residual** (actual − predicted, signed) shows systematic bias: near 0 is
  unbiased; positive means typical underprediction.
- **±10% hit rate** is the share of test houses predicted within ±10% of actual —
  the most intuitive "how often are we close" number (Week 7: GBM 54.5%, RF 56.8%).


In [2]:
def metrics(y, p):
    resid = y - p
    ape = np.abs(resid) / y * 100
    return {
        'R2': r2_score(y, p),
        'RMSE': float(np.sqrt(mean_squared_error(y, p))),
        'MAE': mean_absolute_error(y, p),
        'MAPE': float(np.mean(ape)),
        'MdAPE': float(np.median(ape)),
        'median_residual': float(np.median(resid)),
        'mean_residual': float(np.mean(resid)),
        'hit_10pct': float(np.mean(ape <= 10) * 100),
        'n': len(y),
    }

rows = []
for name, p in preds.items():
    m = metrics(y_test, p)
    rows.append({'model': name, **m})
    print(f"{name}: R2 {m['R2']:.4f} | RMSE ${m['RMSE']:,.0f} | MAE ${m['MAE']:,.0f} | "
          f"MAPE {m['MAPE']:.2f}% | MdAPE {m['MdAPE']:.2f}% | median resid ${m['median_residual']:,.0f} | "
          f"±10% hit {m['hit_10pct']:.1f}%")

summary_tbl = pd.DataFrame(rows).set_index('model')
print()
print(summary_tbl.round(4).to_string())


LightGBM (tuned) — Week 7 champion: R2 0.8911 | RMSE $327,782 | MAE $167,600 | MAPE 13.07% | MdAPE 9.03% | median resid $8,289 | ±10% hit 54.5%
RF (new feats) — Week 6 champion: R2 0.8617 | RMSE $369,380 | MAE $179,843 | MAPE 13.76% | MdAPE 8.33% | median resid $2,161 | ±10% hit 56.8%

                                        R2         RMSE          MAE     MAPE   MdAPE  median_residual  mean_residual  hit_10pct      n
model                                                                                                                                  
LightGBM (tuned) — Week 7 champion  0.8911  327782.3745  167600.0428  13.0747  9.0315        8289.4678     25002.3720    54.4615  12339
RF (new feats) — Week 6 champion    0.8617  369380.2580  179842.8192  13.7617  8.3344        2160.7850     20670.8875    56.7712  12339


## 2. Price-band breakdown (champion only)

One headline MAPE can hide a segment that is systematically off. Bands reuse the
Week 5–7 luxury cut (>2M) so the luxury row is directly comparable to Week 7 
(luxury mean residual +$240,637, 6.0% of actual, 13.6% of test):
- Under 500k
- 500k – 1M
- 1M – 2M
- Over 2M (luxury)

Read each band's MdAPE (typical house in that band) next to its MAPE and mean
residual: a band where MAPE >> MdAPE has a few very large errors; a band with a
large signed mean residual is systematically biased in one direction.


In [3]:
CHAMPION = 'LightGBM (tuned) — Week 7 champion'
if CHAMPION not in preds:
    CHAMPION = next(iter(preds))
    print(f'Champion predictions file missing, evaluating {CHAMPION} instead')
p_champ = preds[CHAMPION]
resid = y_test - p_champ
ape = np.abs(resid) / y_test * 100


In [6]:
# ---- price bands (same >$2M luxury cut as Weeks 5-7) ----
bands = pd.cut(y_test,
               bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
               labels=['< $500k', '$500k-$1M', '$1M-$2M', '> $2M (luxury)'])

band_rows = []
for band in bands.categories:
    mask = bands == band
    m = metrics(y_test[mask], p_champ[mask])
    band_rows.append({'band': band, 'share_pct': mask.mean() * 100, **m})

band_tbl = pd.DataFrame(band_rows).set_index('band')
print(band_tbl.round(2).to_string())
print()
print(f"Luxury check vs Week 7 : luxury share should be ~13.6%, "
      f"mean residual ~+$240,637 — got "
      f"{band_tbl.loc['> $2M (luxury)', 'share_pct']:.1f}% / "
      f"${band_tbl.loc['> $2M (luxury)', 'mean_residual']:,.0f}")


                share_pct    R2       RMSE        MAE   MAPE  MdAPE  median_residual  mean_residual  hit_10pct     n
band                                                                                                                
< $500k             14.13 -1.74  126234.76   68826.08  18.86  10.33        -18204.14      -40007.32      49.03  1744
$500k-$1M           41.83  0.23  121442.63   80303.28  10.86   7.43         -1231.73      -18986.98      62.36  5162
$1M-$2M             30.46  0.13  256199.39  176465.68  12.42   9.54         39599.68       19542.70      52.41  3759
> $2M (luxury)      13.57  0.62  763200.24  519787.45  15.33  12.30        217153.92      240637.29      40.38  1674

Luxury check vs Week 7 : luxury share should be ~13.6%, mean residual ~+$240,637 — got 13.6% / $240,637


## 3. Read the result — what Week 8 adds to Week 7


1. **Typical vs average:** is MdAPE well below MAPE for the champion? yes, the
   typical house is predicted better than the headline MAPE suggests.
   
    For the champion, MdAPE (9.03%) sits well below MAPE (13.07%), a 4.04 pp gap. 
    The typical house is predicted better than the headline MAPE suggests; 
    the mean is dragged up by a long tail of large errors.
    The same gap appears for RF-new (MdAPE 8.33% vs MAPE 13.76%).)
   
2. **Champion vs RF-new on the robust metric:** does the GBM still win on MdAPE,
   or was its MAPE win carried by the tail? No — the GBM does **not** win onMdAPE. 
   
   RF-new's MdAPE (8.33%) beats the champion's (9.03%), and RF-new also has
    the smaller median residual (2,161 vs 8,289) and the higher ±10% hit rate
    (56.8% vs 54.5%). The champion's wins on R² (0.8911 vs 0.8617), RMSE (327,782
    vs 369,380), MAE and MAPE were carried by the tail — it makes fewer very large
    errors — not by better accuracy on the typical house. 
   
3. **Where it breaks:** which price band has the worst MdAPE and the largest
   signed mean residual? 
   Two opposite failure modes at the two ends of the price range:
    - Luxury (>2M, 13.6% of test): worst MdAPE (12.30%), largest dollar errors
        (MAE 519,787), lowest hit rate (40.4%), and the largest signed mean residual
        (+240,637) — systematic underprediction.
    - Cheapest (<500k, 14.1% of test): worst MAPE (18.86%) and a negative
        within-band R² (−1.74, worse than predicting the band mean), with a mean
        residual of −40,007 — systematic overprediction.
    - The middle band (500k–1M, 41.8% of test) is where the model works best:
        MAPE 10.86%, MdAPE 7.43%, hit rate 62.4%, median residual −1,232.


**In one sentence: the model shrinks predictions toward the middle — cheap houses
are predicted too high, expensive houses too low.**

## 4. Save artifacts + summary

Saved to `output_week8/`:
- `metrics_summary.csv` — 1 metrics table (both models) plus  price-band table
- `week8_summary.txt` — the one-page summary to quote in the report

This notebook changes nothing about the champion: it evaluates the Week 7
decision, it does not re-make it.


In [7]:
summary_tbl.reset_index().to_csv(OUT_DIR / 'metrics_summary.csv', index=False)
band_tbl.reset_index().to_csv(OUT_DIR / 'price_band_breakdown.csv', index=False)

champ = summary_tbl.loc[CHAMPION]
summary = [
    'Week 8 — Evaluation Expansion (California close-price prediction)',
    f'Date: {pd.Timestamp.now().date()}',
    f"Champion evaluated: tuned LightGBM (Week 7), test n = {len(y_test)}",
    f"Champion: R2 {champ['R2']:.4f} | RMSE ${champ['RMSE']:,.0f} | MAE ${champ['MAE']:,.0f} | "
    f"MAPE {champ['MAPE']:.2f}% | MdAPE {champ['MdAPE']:.2f}% | ±10% hit {champ['hit_10pct']:.1f}%",
    '',
    'Price-band breakdown (champion):',
    band_tbl.round(2).to_string(),
    '',
    'Scope: handbook Week 8 only (metrics beyond R2 + price-band breakdown). '
    'No retraining, no tuning, no new champion selection; saved Week 6/7 test '
    'predictions reused for descriptive evaluation only.',
]
(OUT_DIR / 'week8_summary.txt').write_text('\n'.join(summary))
print('\n'.join(summary))
print()
print(f'saved -> {OUT_DIR}/ (metrics_summary.csv, price_band_breakdown.csv, week8_summary.txt)')


Week 8 — Evaluation Expansion (California close-price prediction)
Date: 2026-11-08
Champion evaluated: tuned LightGBM (Week 7), test n = 12339
Champion: R2 0.8911 | RMSE $327,782 | MAE $167,600 | MAPE 13.07% | MdAPE 9.03% | ±10% hit 54.5%

Price-band breakdown (champion):
                share_pct    R2       RMSE        MAE   MAPE  MdAPE  median_residual  mean_residual  hit_10pct     n
band                                                                                                                
< $500k             14.13 -1.74  126234.76   68826.08  18.86  10.33        -18204.14      -40007.32      49.03  1744
$500k-$1M           41.83  0.23  121442.63   80303.28  10.86   7.43         -1231.73      -18986.98      62.36  5162
$1M-$2M             30.46  0.13  256199.39  176465.68  12.42   9.54         39599.68       19542.70      52.41  3759
> $2M (luxury)      13.57  0.62  763200.24  519787.45  15.33  12.30        217153.92      240637.29      40.38  1674

Scope: handbook Week 8 o